In [0]:
from pyspark.sql.functions import lit, current_timestamp, to_date, col

In [0]:
metadata_df = (
    spark.table("adbrag.project4_schema.file_metadata")
         .filter("is_active = true")
)

In [0]:
for row in metadata_df.collect():
    source_name = row["source_name"]
    source_file_path = row["source_file_path"]
    target_table = row["target_table"]
    file_format = row["file_format"]
    delimiter = row["delimiter"]
    has_header = row["has_header"]
    source_year = row["source_year"]
    column_names = [c.strip() for c in row["column_names"].split(",")]

    df = (
        spark.read.format(file_format)
        .option("header", str(has_header).lower())
        .option("delimiter", delimiter)
        .option("quote", '"')
        .load(source_file_path)
    )

    df = df.toDF(*column_names)

    df = (
        df.withColumn("SalesOrderLineNumber", col("SalesOrderLineNumber").cast("int"))
          .withColumn("OrderDate", to_date(col("OrderDate"), "yyyy-MM-dd"))
          .withColumn("OrderQuantity", col("OrderQuantity").cast("int"))
          .withColumn("UnitPrice", col("UnitPrice").cast("double")) #ROUND or DECIMAL(10,2) 
          .withColumn("TaxAmount", col("TaxAmount").cast("double"))
          .withColumn("SourceYear", lit(source_year))
          .withColumn("SourceName", lit(source_name))
          .withColumn("LoadTimestamp", current_timestamp())
    )

    df.write.format("delta") \
        .mode("append") \
        .saveAsTable(f"adbrag.project4_schema.{target_table}")

    print(f"Loaded {source_name} into adbrag.project4_schema.{target_table}")